In [2]:
import os
import pandas as pd
import numpy as np
import gc

# Setup Path
base_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_path = os.path.join(base_path, 'data')
sub_path = os.path.join(base_path, 'submissions')

# Load Data - Gunakan dtypes agar lebih ringan sejak awal
df_train = pd.read_csv(os.path.join(data_path, 'train.csv'), low_memory=False, parse_dates=['Timestamp'])
df_test = pd.read_csv(os.path.join(data_path, 'test.csv'), low_memory=False, parse_dates=['Timestamp'])
test_ids = df_test["Trip_ID"].copy()

print(f"Data Loaded. Train: {df_train.shape}, Test: {df_test.shape}")

Data Loaded. Train: (8000000, 25), Test: (4000000, 24)


In [3]:
def preprocess_data(df):
    # 1. Handling Timestamp - Tambahkan Jam dan Hari
    df['Timestamp'] = df['Timestamp'].fillna(pd.Timestamp('2024-01-01 00:00:00'))
    df['Hour'] = df['Timestamp'].dt.hour.astype(np.int8)
    df['DayOfWeek'] = df['Timestamp'].dt.dayofweek.astype(np.int8) # Fitur Baru
    
    # 2. Battery Level Cleaning yang lebih robust
    df["Battery_Level"] = df["Battery_Level"].astype(str).str.replace("%", "", regex=False)
    df["Battery_Level"] = pd.to_numeric(df["Battery_Level"], errors='coerce')
    
    # Isi dengan median (hitung median hanya dari data yang tidak null)
    median_val = df["Battery_Level"].median()
    df["Battery_Level"] = df["Battery_Level"].fillna(median_val).astype(np.float32)
    
    # 3. Drop Trip_ID di Train (tapi simpan di Test untuk nanti)
    cols_to_drop = ['Timestamp']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
    return df

df_train = preprocess_data(df_train)
df_test = preprocess_data(df_test)

# Kategorikal Handling
cat_cols = ["Pickup_Zone", "Dropoff_Zone", "Car_Model", "Device_FP", 
            "Promo_Code", "Payment_Method", "Weather", "Traffic", "Signal_Strength"]

for col in cat_cols:
    df_train[col] = df_train[col].astype(str).fillna("Unknown").replace(["nan", "None", "NaN"], "Unknown")
    df_test[col] = df_test[col].astype(str).fillna("Unknown").replace(["nan", "None", "NaN"], "Unknown")

# Optimasi Numerik ke Float32
num_cols = df_train.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    if col != 'Trip_Label':
        df_train[col] = df_train[col].astype(np.float32)
        if col in df_test.columns:
            df_test[col] = df_test[col].astype(np.float32)

gc.collect() # Bersihkan RAM
print(f"Preprocessing Selesai. Train: {len(df_train)}")

Preprocessing Selesai. Train: 8000000


In [4]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

# Gunakan 25% data (Sekitar 2 Juta Baris)
df_sample = df_train.sample(frac=0.25, random_state=42)

X = df_sample.drop(columns=['Trip_ID', 'Trip_Label'], errors='ignore')
y = df_sample['Trip_Label']

x_train, x_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Hapus df_sample dari memori untuk lega-in RAM saat training
del df_sample
gc.collect()

model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,      # Sedikit dinaikkan agar konvergensi lebih cepat (hemat waktu)
    depth=6,                 # Turunkan ke 6 (sangat membantu meringankan beban RAM)
    loss_function="MultiClass",
    eval_metric="TotalF1", 
    auto_class_weights='Balanced', 
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    
    rsm=0.5,                 # Hanya ambil 50% fitur tiap split (RAM makin lega)
    max_bin=128,             # Mengurangi jumlah histogram numerik di memori
    bootstrap_type='Bernoulli', 
    subsample=0.5            # Hanya ambil 50% baris untuk tiap tree pembangunan
)

model.fit(
    x_train, y_train,
    cat_features=cat_cols,
    eval_set=(x_val, y_val),
    use_best_model=True
)

0:	learn: 0.4429342	test: 0.4433986	best: 0.4433986 (0)	total: 3.44s	remaining: 1h 54m 37s
100:	learn: 0.5647404	test: 0.5645555	best: 0.5648512 (92)	total: 4m	remaining: 1h 15m 29s
200:	learn: 0.5755437	test: 0.5740502	best: 0.5740502 (200)	total: 8m 9s	remaining: 1h 13m 2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5742019891
bestIteration = 237

Shrink model to first 238 iterations.


In [5]:
# Pastikan fitur test sama dengan fitur train
feature_names = X.columns.tolist()
x_test = df_test[feature_names]

print(f"Sedang memprediksi {len(x_test)} baris...")
preds = model.predict(x_test)

submission = pd.DataFrame({
    "Trip_ID": test_ids,
    "Trip_Label": preds.flatten()
})

if not os.path.exists(sub_path):
    os.makedirs(sub_path)

output_file = os.path.join(sub_path, "submission_f0.25_lr0.03.csv")
submission.to_csv(output_file, index=False)
print(f"Selesai! File disimpan di: {output_file}")

Sedang memprediksi 4000000 baris...
Selesai! File disimpan di: /home/mmarufqk/DSMLComp/Ctrl+Alt+Del_DataVers_AnavaUGM/submissions/submission_f0.25_lr0.03.csv
